# Step 3 — Train Model

Two-phase training:
- **Phase 1**: Frozen ResNet50 backbone, train only the 7 regression heads (lr=1e-3, 10 epochs max).
- **Phase 2**: Unfreeze backbone, full fine-tune at a much lower lr (1e-5, 15 epochs max).

**Run from repo root:** `airsight/`

## 3.1 — Imports & sys.path

In [ ]:
import sys, os
from pathlib import Path

# Ensure src/ is importable regardless of kernel CWD
REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import tensorflow as tf
from dataset import make_dataset
from model import build_model, TARGET_COLS

print(f"TensorFlow  : {tf.__version__}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Target cols : {TARGET_COLS}")

## 3.2 — Build tf.data pipelines

In [ ]:
BATCH_SIZE = 32

train_ds = make_dataset(REPO_ROOT / "data" / "train.csv", batch_size=BATCH_SIZE, training=True)
val_ds   = make_dataset(REPO_ROOT / "data" / "val.csv",   batch_size=BATCH_SIZE, training=False)
test_ds  = make_dataset(REPO_ROOT / "data" / "test.csv",  batch_size=BATCH_SIZE, training=False)

print(f"Train batches : {len(train_ds)}")
print(f"Val   batches : {len(val_ds)}")
print(f"Test  batches : {len(test_ds)}")

# Verify one batch shape + label keys
for images, labels in train_ds.take(1):
    print(f"\nBatch image shape : {images.shape}")
    print(f"Label keys        : {list(labels.keys())}")
    for k, v in labels.items():
        print(f"  {k:<8s} shape={v.shape}  dtype={v.dtype}")

## 3.3 — Instantiate model & define callbacks

In [ ]:
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

model = build_model(learning_rate=1e-3)
model.summary(line_length=80, show_trainable=True)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS_DIR / "best_model.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
]

print("\nCallbacks ready.")
print(f"Checkpoint path : {ARTIFACTS_DIR / 'best_model.keras'}")

## 3.4 — Phase 1: Frozen backbone training (max 10 epochs, lr=1e-3)

In [ ]:
print("=" * 60)
print("PHASE 1 — Frozen backbone  |  lr=1e-3  |  max 10 epochs")
print("=" * 60)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1,
)

print(f"\nPhase 1 complete.")
print(f"  Epochs run   : {len(history_phase1.epoch)}")
print(f"  Best val_loss: {min(history_phase1.history['val_loss']):.4f}")

## 3.5 — Phase 2: Unfreeze & fine-tune (max 15 epochs, lr=1e-5)

In [ ]:
print("=" * 60)
print("PHASE 2 — Full fine-tune  |  lr=1e-5  |  max 15 epochs")
print("=" * 60)

# Unfreeze the backbone
backbone = model.get_layer("resnet50")
backbone.trainable = True

# Recompile with the same loss/metrics but a much lower LR
# NOTE: recompile is MANDATORY after changing trainable flags in TF 2.x
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss={col: "mse" for col in TARGET_COLS},
    metrics={col: ["mae"] for col in TARGET_COLS},
)

trainable_after = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f"Trainable params after unfreeze : {trainable_after:,}")

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
    verbose=1,
)

print(f"\nPhase 2 complete.")
print(f"  Epochs run   : {len(history_phase2.epoch)}")
print(f"  Best val_loss: {min(history_phase2.history['val_loss']):.4f}")

## 3.6 — Final evaluation on test set

In [ ]:
print("=" * 60)
print("FINAL EVALUATION on test_ds")
print("=" * 60)

results = model.evaluate(test_ds, verbose=1, return_dict=True)

print("\n-- Per-pollutant Test MAE (scaled units) --")
for col in TARGET_COLS:
    mae_key = f"{col}_mae"
    mse_key = f"{col}_loss"
    mae = results.get(mae_key, float("nan"))
    mse = results.get(mse_key, float("nan"))
    print(f"  {col:<8s}  MAE={mae:.4f}   MSE={mse:.4f}")

print(f"\n  Aggregate loss (sum of MSEs) : {results['loss']:.4f}")
print(f"\nBest model saved to: {ARTIFACTS_DIR / 'best_model.keras'}")
print("\nStep 3 complete. Proceed to notebook 04_evaluate.ipynb")